# Feedforward, integral action, cascaded loops and windup: notebook

The chapter's robot: mass m = 8 kg, rolling drag b = 2 N·s/m, drive force limited to ±4 N, controller at 100 Hz. The speed loop is RO-060's PI controller, kp = 20 N per m/s and ki = 10 N per m. The heading starts from RO-060's PID (0.2, 0.283, 0.05). The heading loop is a cascade: an outer P loop on the heading gives a yaw-rate command, an inner PI loop on the gyro's yaw rate gives the torque (limited to ±0.2 N·m). Every number in the Note is computed here. The model and the controllers are in `images/robotctl.py`, which every figure script also uses. Nothing here is random.

In [1]:
import sys
sys.path.insert(0, "images")
import numpy as np
from robotctl import *

print("speed loop poles:", np.roots([M, B + KP, KI]).round(3), " inner heading loop poles:", np.roots([I_YAW, KW, KIW]).round(3))
print("largest acceleration from rest:", U_MAX / M, "m/s^2;  left at 0.5 m/s:", (U_MAX - B * 0.5) / M, "m/s^2")

speed loop poles: [-2.175 -0.575]  inner heading loop poles: [-10. -10.]
largest acceleration from rest: 0.5 m/s^2;  left at 0.5 m/s: 0.375 m/s^2


## 1. Feedback alone trails the plan (Note, Sections 1 and 2)

The planner's speed-up: from rest at 0.25 m/s² to 0.5 m/s in 2 s, then cruise.

In [2]:
def report(name, s, T=12):
    t, v, r = s["t"], s["v"], s["r"]
    e = r - v
    gap = np.cumsum(r - v) * DT
    i2 = int(round(2 / DT))
    print(f"{name:28s} largest speed error {e.max():.4f} m/s at t = {t[e.argmax()]:.2f} s; error at 2 s {e[i2]:.4f};"
          f" gap behind the plan at 12 s {gap[-1]:.4f} m; speed at 12 s {v[-1]:.4f}; largest force {s['u'].max():.3f} N")
    return s

fb = report("feedback only", speed_loop(12, profile, aref=profile_acc))
both = report("feedforward + feedback", speed_loop(12, profile, aref=profile_acc, ff=True))
print("gap = integral term / ki at cruise:", B * 0.5 / KI, "m")
gfb = np.cumsum(fb["r"] - fb["v"]) * DT
print(f"feedback alone: largest gap {gfb.max() * 100:.1f} cm at t = {fb['t'][gfb.argmax()]:.2f} s")
print("feedforward force at t = 1 s:", M * 0.25 + B * 0.25, "N;  at the end of the speed-up:", M * 0.25 + B * 0.5, "N")

feedback only                largest speed error 0.0845 m/s at t = 1.10 s; error at 2 s 0.0762; gap behind the plan at 12 s 0.1003 m; speed at 12 s 0.5002; largest force 2.946 N
feedforward + feedback       largest speed error 0.0000 m/s at t = 0.13 s; error at 2 s -0.0000; gap behind the plan at 12 s -0.0000 m; speed at 12 s 0.5000; largest force 2.995 N
gap = integral term / ki at cruise: 0.1 m
feedback alone: largest gap 15.8 cm at t = 2.50 s
feedforward force at t = 1 s: 2.5 N;  at the end of the speed-up: 3.0 N


On carpet the real drag is 3 N·s/m, but the feedforward still uses the model's 2 N·s/m.

In [3]:
fbc = report("carpet: feedback only", speed_loop(12, profile, aref=profile_acc, b_true=3))
ffc = report("carpet: feedforward only", speed_loop(12, profile, aref=profile_acc, ff=True, kp=0, ki=0, b_true=3))
bothc = report("carpet: feedforward + feedback", speed_loop(12, profile, aref=profile_acc, ff=True, b_true=3))
print("feedforward only settles at b_model v / b_true =", B * 0.5 / 3, "m/s")
print("carpet gap = extra drag force / ki:", (3 - B) * 0.5 / KI, "m")

carpet: feedback only        largest speed error 0.0903 m/s at t = 1.38 s; error at 2 s 0.0881; gap behind the plan at 12 s 0.1502 m; speed at 12 s 0.5001; largest force 3.277 N
carpet: feedforward only     largest speed error 0.1639 m/s at t = 12.00 s; error at 2 s 0.0493; gap behind the plan at 12 s 1.3970 m; speed at 12 s 0.3361; largest force 2.995 N
carpet: feedforward + feedback largest speed error 0.0146 m/s at t = 2.23 s; error at 2 s 0.0140; gap behind the plan at 12 s 0.0498 m; speed at 12 s 0.4999; largest force 3.404 N
feedforward only settles at b_model v / b_true = 0.3333333333333333 m/s
carpet gap = extra drag force / ki: 0.05 m


## 2. State feedback, the integral state and the internal model (Note, Section 3)

State feedback on the speed: u = −k v + kf r with k = 20 and kf = k + b = 22 from the model.

In [4]:
print("ramp force on 1 degree:", round(RAMP_1DEG, 4), "N")
for name, kw in [("model exact", {}), ("carpet b = 3", dict(b_true=3)), ("1 degree ramp", dict(dist=lambda t: -RAMP_1DEG))]:
    s0 = speed_loop(15, lambda t: 0.5, v0=0.5, i0=0.0, sf=(20, 22), ki=0, **kw)
    s1 = speed_loop(15, lambda t: 0.5, v0=0.5, i0=0.0, sf=(20, 22), **kw)
    late = np.where(np.abs(s1["v"] - 0.5) > 0.005)[0]
    print(f"{name:14s} state feedback settles at {s0['v'][-1]:.4f} m/s;  with the integral state {s1['v'][-1]:.4f} m/s,"
          f" lowest {s1['v'].min():.4f}, within 0.005 m/s after {s1['t'][late[-1]] + DT if len(late) else 0:.2f} s, integral term {s1['i'][-1]:.4f} N")
print("formulas: carpet", 22 * 0.5 / 23, " ramp", (11 - RAMP_1DEG) / 22)

ramp force on 1 degree: 1.3697 N
model exact    state feedback settles at 0.5000 m/s;  with the integral state 0.5000 m/s, lowest 0.5000, within 0.005 m/s after 0.00 s, integral term 0.0000 N
carpet b = 3   state feedback settles at 0.4783 m/s;  with the integral state 0.5000 m/s, lowest 0.4827, within 0.005 m/s after 3.61 s, integral term 0.4998 N
1 degree ramp  state feedback settles at 0.4377 m/s;  with the integral state 0.5000 m/s, lowest 0.4509, within 0.005 m/s after 5.32 s, integral term 1.3693 N
formulas: carpet 0.4782608695652174  ramp 0.4377425064909995


The out-of-round wheel at the docking creep speed of 0.05 m/s: the wheel (radius 0.05 m) turns at 1 rad/s and adds a force 0.5 sin(t) N.

In [5]:
w0 = 0.05 / 0.05
print("wheel turn rate:", w0, "rad/s, period", round(2 * np.pi / w0, 2), "s")
print("PI ripple from the formula: 0.5 / |8j + 2 + 20 - 10j| =", 0.5 / abs(8j * w0 + 2 + KP + KI / (1j * w0)))
d = lambda t: 0.5 * np.sin(w0 * t)
pi_ = speed_loop(60, lambda t: 0.05, v0=0.05, ff=True, i0=0.0, dist=d)
rs = speed_loop(60, lambda t: 0.05, v0=0.05, ff=True, i0=0.0, dist=d, res=RES)
late = pi_["t"] >= 40
print("PI only: speed swings", pi_["v"][late].min().round(4), "to", pi_["v"][late].max().round(4), "m/s, amplitude",
      ((pi_["v"][late].max() - pi_["v"][late].min()) / 2).round(4))
print("PI + resonator: amplitude", ((rs["v"][late].max() - rs["v"][late].min()) / 2), " resonator output amplitude",
      np.abs(rs["ures"][late]).max().round(4), "N")
for T0 in (0, 10, 20):
    w = (rs["t"] >= T0) & (rs["t"] < T0 + 2 * np.pi)
    print(f"  PI + resonator, wheel turn starting at {T0} s: speed amplitude {(rs['v'][w].max() - rs['v'][w].min()) / 2:.5f} m/s")
P = np.poly1d([M, B]); s_ = np.poly1d([1, 0]); R = np.poly1d([1, 0, RES[0] ** 2])
print("closed-loop poles with the resonator:", (P * s_ * R + np.poly1d([KP, KI]) * R + s_ * np.poly1d([RES[2], RES[1]])).roots.round(3))

wheel turn rate: 1.0 rad/s, period 6.28 s
PI ripple from the formula: 0.5 / |8j + 2 + 20 - 10j| = 0.022633936510629633
PI only: speed swings 0.0273 to 0.0727 m/s, amplitude 0.0227
PI + resonator: amplitude 5.387010282298377e-14  resonator output amplitude 0.5 N
  PI + resonator, wheel turn starting at 0 s: speed amplitude 0.00965 m/s
  PI + resonator, wheel turn starting at 10 s: speed amplitude 0.00002 m/s
  PI + resonator, wheel turn starting at 20 s: speed amplitude 0.00000 m/s
closed-loop poles with the resonator: [-0.688+1.187j -0.688-1.187j -0.687+0.438j -0.687-0.438j]


## 3. Held for 3 s: windup and anti-windup (Note, Sections 4 and 5)

Cruising at 0.5 m/s with feedforward plus PI (the integral term starts at 0 N because the feedforward supplies the 1 N of drag). A person holds the robot still from t = 2 s to t = 5 s.

In [6]:
def held(mode="none", kaw=None):
    return speed_loop(14, lambda t: 0.5, v0=0.5, ff=True, i0=0.0, hold=(2, 5), mode=mode, kaw=kaw)

rows = {}
for name, mode, kaw in [("no anti-windup", "none", None), ("clamping", "clamp", None),
                        ("back-calculation kaw = 1/s", "bc", 1.0), ("back-calculation kaw = 2/s", "bc", 2.0),
                        ("back-calculation kaw = ki/kp", "bc", KI / KP)]:
    s = held(mode, kaw)
    t, v = s["t"], s["v"]
    i5 = int(round(5 / DT))
    late = np.where(np.abs(v - 0.5) > 0.01)[0]
    rows[name] = s
    print(f"{name:30s} at release: command {s['c'][i5 - 1]:6.2f} N, integral {s['i'][i5 - 1]:6.2f} N; peak speed {v[i5:].max():.3f} m/s"
          f" at {t[i5 + v[i5:].argmax()]:.2f} s; above 0.6 m/s for {(v > 0.6).sum() * DT:.2f} s; within 0.01 m/s after {t[late[-1]] + DT:.2f} s")
s = rows["no anti-windup"]
k = 500 + np.argmax(s["v"][500:] >= 0.5)
j = 500 + np.argmax(s["u"][500:] < U_MAX - 1e-9)
print(f"no anti-windup: passes 0.5 m/s at {s['t'][k]:.2f} s with the integral at {s['i'][k]:.2f} N; the drive leaves 4 N at {s['t'][j]:.2f} s, speed then {s['v'][j]:.3f} m/s")
for name in ("clamping", "back-calculation kaw = 1/s"):
    s = rows[name]
    j = 500 + np.argmax(s["u"][500:] < U_MAX - 1e-9)
    k = 500 + np.argmax(s["v"][500:] >= 0.5)
    print(f"{name}: the drive leaves 4 N at {s['t'][j]:.2f} s at {s['v'][j]:.3f} m/s (integral {s['i'][j]:.2f} N); passes 0.5 m/s at {s['t'][k]:.2f} s with the integral at {s['i'][k]:.3f} N")
print("integral growth while held:", KI * 0.5, "N per s;  ideal command at release = 1 + kp 0.5 + 3 s of growth =", 1 + KP * 0.5 + KI * 0.5 * 3)
print("back-calculation balance while held, kaw = 1/s: command = 4 + ki e / kaw =", 4 + KI * 0.5 / 1.0, "N; integral = command - 1 - 10 =", 4 + KI * 0.5 / 1.0 - 1 - KP * 0.5, "N")
print("tracking time constant kp/ki =", KP / KI, "s")

no anti-windup                 at release: command  25.90 N, integral  14.90 N; peak speed 1.039 m/s at 8.07 s; above 0.6 m/s for 5.09 s; within 0.01 m/s after 14.01 s
clamping                       at release: command  11.00 N, integral   0.00 N; peak speed 0.515 m/s at 7.42 s; above 0.6 m/s for 0.00 s; within 0.01 m/s after 8.62 s
back-calculation kaw = 1/s     at release: command   9.10 N, integral  -1.90 N; peak speed 0.500 m/s at 14.00 s; above 0.6 m/s for 0.00 s; within 0.01 m/s after 7.43 s
back-calculation kaw = 2/s     at release: command   6.51 N, integral  -4.49 N; peak speed 0.499 m/s at 14.00 s; above 0.6 m/s for 0.00 s; within 0.01 m/s after 10.34 s


back-calculation kaw = ki/kp   at release: command  13.33 N, integral   2.33 N; peak speed 0.594 m/s at 6.98 s; above 0.6 m/s for 0.00 s; within 0.01 m/s after 11.39 s
no anti-windup: passes 0.5 m/s at 6.15 s with the integral at 17.71 N; the drive leaves 4 N at 7.82 s, speed then 1.013 m/s
clamping: the drive leaves 4 N at 5.77 s at 0.351 m/s (integral 0.00 N); passes 0.5 m/s at 6.60 s with the integral at 0.428 N
back-calculation kaw = 1/s: the drive leaves 4 N at 5.63 s at 0.292 m/s (integral -1.18 N); passes 0.5 m/s at 5.00 s with the integral at -1.901 N
integral growth while held: 5.0 N per s;  ideal command at release = 1 + kp 0.5 + 3 s of growth = 26.0
back-calculation balance while held, kaw = 1/s: command = 4 + ki e / kaw = 9.0 N; integral = command - 1 - 10 = -2.0 N
tracking time constant kp/ki = 2.0 s


## 4. At the dock: motion control against force control (Note, Section 6)

Creeping at 0.05 m/s, the robot touches the dock at t = 2 s and stays still; the dock pushes back with the drive force.

In [7]:
for mode in ("none", "clamp"):
    s = speed_loop(40, lambda t: 0.05, v0=0.05, ff=True, i0=0.0, hold=(2, 1e9), mode=mode)
    j = np.argmax(s["u"] >= U_MAX - 1e-9)
    print(f"motion control, {mode:5s}: force just after contact {s['u'][201]:.2f} N, full 4 N at t = {s['t'][j]:.2f} s,"
          f" command after 30 s at the dock {s['c'][3200]:.2f} N (integral {s['i'][3200]:.2f} N)")

for ctrl in ("none", "clamp", "force"):
    s = dock_run(100, ctrl)
    t = s["t"]
    left = np.where((t >= 32) & (s["x"] < 0.1 - 1e-6))[0]
    print(f"{ctrl:5s}: touches at t = {t[np.argmax(s['f'] > 0)]:.2f} s, force at t = 20 s {s['f'][2000]:.2f} N,"
          f" leaves the dock {t[left[0]] - 32:.2f} s after the back-away command")
print("integral growth at the dock:", KI * 0.05, "N per s")
print("force control: drive force = wanted contact force = 2 N (at rest, m a = u - b v - F_dock with a = v = 0)")

motion control, none : force just after contact 1.10 N, full 4 N at t = 7.81 s, command after 30 s at the dock 16.10 N (integral 15.00 N)


motion control, clamp: force just after contact 1.10 N, full 4 N at t = 7.81 s, command after 30 s at the dock 4.00 N (integral 2.90 N)


none : touches at t = 1.99 s, force at t = 20 s 4.00 N, leaves the dock 27.85 s after the back-away command


clamp: touches at t = 1.99 s, force at t = 20 s 4.00 N, leaves the dock 3.66 s after the back-away command
force: touches at t = 1.99 s, force at t = 20 s 2.00 N, leaves the dock 0.01 s after the back-away command
integral growth at the dock: 0.5 N per s
force control: drive force = wanted contact force = 2 N (at rest, m a = u - b v - F_dock with a = v = 0)


## 5. Rate limits and deadband: the velocity smoother (Note, Section 7)

Nav2's velocity smoother, open loop, at 20 Hz, accelerating and braking at 0.25 m/s², top speed 0.6 m/s, deadband 0.02 m/s. The planner's raw commands: 0, then 0.5 m/s from 0.5 s, 0.015 m/s from 3.5 s, 0 from 7 s.

In [8]:
f, T = 20, 9
tt = np.arange(0, T, 1 / f)
raw = np.where(tt < 0.5, 0, np.where(tt < 3.5, 0.5, np.where(tt < 7.0, 0.015, 0.0)))
pub, inner = smoother(raw)
print("step per cycle:", A_SMOOTH / f, "m/s")
print("published:", list(zip(tt[9:13].round(2), pub[9:13].round(4))))
print("reaches 0.5 m/s at", tt[np.argmax(pub >= 0.5 - 1e-9)], "s; inner value reaches 0.015 at",
      tt[np.argmax((tt > 3.5) & (np.abs(inner - 0.015) < 1e-9))], "s; published value then", pub[np.argmax((tt > 3.5) & (np.abs(inner - 0.015) < 1e-9))])
hold_cmd = lambda arr: (lambda t: arr[min(int(t * f + 1e-9), len(arr) - 1)])
acc = np.where(pub != 0, np.diff(np.r_[0, inner]) * f, 0.0)   # acceleration of the command actually sent
for name, arr, a in [("raw commands", raw, None), ("smoothed commands", pub, acc)]:
    s = speed_loop(T - 0.01, hold_cmd(arr), aref=(hold_cmd(a) if a is not None else None), ff=True, i0=0.0, mode="clamp")
    print(f"{name:18s}: largest force asked {np.abs(s['c']).max():.2f} N, time at +4 N {(s['u'] >= U_MAX - 1e-9).sum() * DT:.2f} s, at -4 N {(s['u'] <= -U_MAX + 1e-9).sum() * DT:.2f} s,"
          f" top speed reached {s['v'].max():.3f} m/s")

step per cycle: 0.0125 m/s
published: [(np.float64(0.45), np.float64(0.0)), (np.float64(0.5), np.float64(0.0)), (np.float64(0.55), np.float64(0.025)), (np.float64(0.6), np.float64(0.0375))]
reaches 0.5 m/s at 2.45 s; inner value reaches 0.015 at 5.4 s; published value then 0.0
raw commands      : largest force asked 11.00 N, time at +4 N 0.77 s, at -4 N 0.48 s, top speed reached 0.515 m/s


smoothed commands : largest force asked 3.10 N, time at +4 N 0.00 s, at -4 N 0.00 s, top speed reached 0.509 m/s


## 6. The heading cascade (Note, Section 8)

RO-060's heading PID against the cascade: outer P (1 rad/s per rad, capped at 1 rad/s), inner PI on the gyro (2 and 10).

In [9]:
carpet = lambda t: -0.05 if t >= 1 else 0.0
p = heading_pid(30, lambda t: 0.0, tau_d=carpet)
k = np.where(np.abs(np.degrees(p["th"])) > 0.5)[0]
print(f"RO-060 PID, carpet: heading {np.degrees(p['th'].min()):.2f} degrees at {p['t'][p['th'].argmin()]:.2f} s, back within 0.5 degrees at {p['t'][k[-1]] + DT:.2f} s")
c = heading_loop(30, lambda t: 0.0, tau_d=carpet)
k = np.where(np.abs(c["w"]) > 0.01)[0]
print(f"cascade, carpet: heading {np.degrees(c['th'].min()):.3f} degrees at {c['t'][c['th'].argmin()]:.2f} s, yaw rate back within 0.01 rad/s at {c['t'][k[-1]] + DT:.2f} s, inner integral {c['i'][-1]:.4f} N m")
q = heading_pid(30, lambda t: 0.0, kp=KW * KTH + KIW, ki=KIW * KTH, kd=KW, tau_d=carpet)
print(f"single PID (12, 10, 2), carpet: heading {np.degrees(q['th'].min()):.3f} degrees")
print("multiplied-out gains: kp =", KW * KTH + KIW, " ki =", KIW * KTH, " kd =", KW)

RO-060 PID, carpet: heading -11.12 degrees at 3.65 s, back within 0.5 degrees at 11.56 s


cascade, carpet: heading -0.212 degrees at 1.37 s, yaw rate back within 0.01 rad/s at 1.21 s, inner integral 0.0500 N m
single PID (12, 10, 2), carpet: heading -0.209 degrees
multiplied-out gains: kp = 12.0  ki = 10.0  kd = 2.0


In [10]:
def turn_report(name, s):
    th, w = s["th"], s["w"]
    j = np.where(np.abs(th - np.pi / 2) > np.radians(2))[0][-1]
    settle = f"within 2 degrees after {s['t'][j + 1]:.2f} s" if j + 1 < len(th) else "not within 2 degrees by the end"
    print(f"{name:34s} peak yaw rate {w.max():.3f} rad/s, overshoot {np.degrees(th.max() - np.pi / 2):7.2f} degrees, {settle}")

turn_report("cascade", heading_loop(30, lambda t: np.pi / 2))
turn_report("RO-060 PID", heading_pid(30, lambda t: np.pi / 2))
turn_report("single PID (12, 10, 2)", heading_pid(30, lambda t: np.pi / 2, kp=12, ki=10, kd=2))
turn_report("single PID (12, 10, 2), clamping", heading_pid(30, lambda t: np.pi / 2, kp=12, ki=10, kd=2, clamp=True))
s = heading_loop(6, lambda t: np.pi / 2)
t, th, w, wc = s["t"], s["th"], s["w"], s["wc"]
i = np.argmax(np.abs(w - wc) < 0.05 * np.abs(wc))
print(f"cascade: yaw rate within 5 percent of its command after {t[i]:.2f} s; torque at its limit for {(np.abs(s['tau']) >= TAU_MAX - 1e-9).sum() * DT:.2f} s;"
      f" command at the 1 rad/s cap for {(np.abs(wc) >= W_CAP - 1e-9).sum() * DT:.2f} s")
print(f"perfect inner loop, formula: 0.571 + ln(1/0.0349) = {0.5708 + np.log(1 / np.radians(2)):.2f} s")

cascade                            peak yaw rate 1.015 rad/s, overshoot   -0.00 degrees, within 2 degrees after 4.22 s
RO-060 PID                         peak yaw rate 0.927 rad/s, overshoot   20.83 degrees, within 2 degrees after 11.01 s


single PID (12, 10, 2)             peak yaw rate 8.681 rad/s, overshoot 1724.29 degrees, not within 2 degrees by the end
single PID (12, 10, 2), clamping   peak yaw rate 2.190 rad/s, overshoot   48.38 degrees, within 2 degrees after 4.60 s
cascade: yaw rate within 5 percent of its command after 0.48 s; torque at its limit for 0.46 s; command at the 1 rad/s cap for 0.82 s
perfect inner loop, formula: 0.571 + ln(1/0.0349) = 3.93 s


In [11]:
# Why the same gains overshoot the turn in one loop but not in the cascade: where each starts braking (torque < 0)
for name, s in [("single PID (12, 10, 2), clamping", heading_pid(30, lambda t: np.pi / 2, kp=12, ki=10, kd=2, clamp=True)),
                ("cascade", heading_loop(30, lambda t: np.pi / 2))]:
    k = np.argmax(s["tau"] < 0)
    print(f"{name:34s} starts braking at t = {s['t'][k]:.2f} s, {np.pi / 2 - s['th'][k]:.2f} rad from the target,"
          f" yaw rate {s['w'][k]:.2f} rad/s")
print("single PID's first request: 12 x 1.5708 =", round(12 * np.pi / 2, 1), "N m; its P and D parts cancel when w = 6 e")
turn_report("cascade without the 1 rad/s cap", heading_loop(30, lambda t: np.pi / 2, wcap=1e9))

single PID (12, 10, 2), clamping   starts braking at t = 1.10 s, 0.35 rad from the target, yaw rate 2.19 rad/s
cascade                            starts braking at t = 0.64 s, 1.18 rad from the target, yaw rate 1.02 rad/s
single PID's first request: 12 x 1.5708 = 18.8 N m; its P and D parts cancel when w = 6 e
cascade without the 1 rad/s cap    peak yaw rate 1.191 rad/s, overshoot   -0.00 degrees, within 2 degrees after 4.19 s
